# 02. 월별 라벨 데이터 전처리 및 통합

2024년 1월부터 2026년 6월까지의 월별 라벨 CSV 30개를 승인된 규칙으로 정리하여 하나의 데이터셋으로 통합한다.

**적용 규칙**

- 원본 CSV는 수정하지 않는다.
- 완전히 비어 있는 행만 제거한다.
- `Unnamed` 컬럼은 제거한다.
- 승인된 컬럼명 변환만 적용한다.
- 날짜·물량·이벤트 자료형을 명시적으로 검증한다.
- 결측값을 임의로 채우지 않는다.
- 중복 날짜가 있으면 저장하지 않고 오류로 중단한다.

**출력**

- Python 분석용: `data/interim/daily_volume_event_clean.parquet`
- Excel·사람 검토용: `data/interim/daily_volume_event_clean.csv`

In [1]:
from pathlib import Path
import hashlib
import re
import unicodedata

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)


def find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "data" / "raw").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. postcast 또는 notebooks 폴더에서 실행하세요."
    )


PROJECT_ROOT = find_project_root()
MONTHLY_DIR = PROJECT_ROOT / "data" / "raw" / "monthly_labels"
OUTPUT_PATH = PROJECT_ROOT / "data" / "interim" / "daily_volume_event_clean.parquet"

print(f"프로젝트 루트: {PROJECT_ROOT}")
print(f"입력 경로: {MONTHLY_DIR}")
print(f"출력 경로: {OUTPUT_PATH}")

프로젝트 루트: /Users/hyewon/Documents/postcast
입력 경로: /Users/hyewon/Documents/postcast/data/raw/monthly_labels
출력 경로: /Users/hyewon/Documents/postcast/data/interim/daily_volume_event_clean.parquet


## 1. 전처리 규칙 정의

컬럼명은 아래의 11개 컬럼으로 통일한다. 원본에 존재하는 공백 차이와 `제1기분 자동차세2`만 승인된 이름으로 변환한다.

In [2]:
ENCODINGS = ("utf-8-sig", "utf-8", "cp949", "euc-kr")

CANONICAL_COLUMNS = [
    "접수일자",
    "접수지역",
    "접수통수",
    "요일",
    "제1기분 자동차세",
    "재산세(건축)",
    "정기분 주민세",
    "주민세(사업소분)",
    "재산세(토지)",
    "제2기분 자동차세",
    "사회보험료 통합",
]

EVENT_COLUMNS = [
    "제1기분 자동차세",
    "재산세(건축)",
    "정기분 주민세",
    "주민세(사업소분)",
    "재산세(토지)",
    "제2기분 자동차세",
    "사회보험료 통합",
]

COLUMN_RENAME = {
    "제1기분 자동차세2": "제1기분 자동차세",
    "재산세 (건축)": "재산세(건축)",
    "주민세 (사업소분)": "주민세(사업소분)",
    "재산세 (토지)": "재산세(토지)",
}

KOREAN_WEEKDAY = {
    0: "월",
    1: "화",
    2: "수",
    3: "목",
    4: "금",
    5: "토",
    6: "일",
}


def nfc(value) -> str:
    return unicodedata.normalize("NFC", str(value)).strip()


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    result.columns = [nfc(column) for column in result.columns]
    return result


def read_csv_auto(path: Path):
    errors = []
    for encoding in ENCODINGS:
        try:
            return pd.read_csv(path, encoding=encoding), encoding
        except UnicodeError as error:
            errors.append(f"{encoding}: {error}")
    raise UnicodeError(f"{path.name} 인코딩 판별 실패: {errors}")


def fully_blank_mask(df: pd.DataFrame) -> pd.Series:
    return df.apply(
        lambda row: all(
            pd.isna(value)
            or (isinstance(value, str) and not value.strip())
            for value in row
        ),
        axis=1,
    )


def file_hash(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def extract_yyyymm(filename: str):
    match = re.search(r"(20\d{4})", nfc(filename))
    return match.group(1) if match else None

## 2. 월별 파일 로드 및 정제

각 파일을 개별적으로 검증한다. 한 파일이라도 최종 스키마, 날짜, 접수통수 또는 이벤트 값이 기준을 위반하면 병합과 저장을 중단한다.

In [3]:
monthly_files = sorted(
    MONTHLY_DIR.rglob("*.csv"),
    key=lambda path: extract_yyyymm(path.name) or nfc(path.name),
)

if len(monthly_files) != 30:
    raise ValueError(f"월별 CSV는 30개여야 합니다. 현재 {len(monthly_files)}개입니다.")

raw_hashes_before = {str(path): file_hash(path) for path in monthly_files}

cleaned_frames = []
processing_rows = []

for path in monthly_files:
    raw_df, encoding = read_csv_auto(path)
    raw_df = normalize_columns(raw_df)

    blank_mask = fully_blank_mask(raw_df)
    df = raw_df.loc[~blank_mask].copy()

    unnamed_columns = [
        column for column in df.columns if column.startswith("Unnamed")
    ]
    df = df.drop(columns=unnamed_columns, errors="ignore")
    df = df.rename(columns=COLUMN_RENAME)

    duplicated_columns = df.columns[df.columns.duplicated()].tolist()
    if duplicated_columns:
        raise ValueError(
            f"{nfc(path.name)}: 컬럼명 통일 후 중복 컬럼 발생: {duplicated_columns}"
        )

    missing_columns = [
        column for column in CANONICAL_COLUMNS if column not in df.columns
    ]
    extra_columns = [
        column for column in df.columns if column not in CANONICAL_COLUMNS
    ]
    if missing_columns or extra_columns:
        raise ValueError(
            f"{nfc(path.name)} 스키마 불일치 - "
            f"누락: {missing_columns}, 추가: {extra_columns}"
        )

    df = df[CANONICAL_COLUMNS].copy()

    df["접수일자"] = pd.to_datetime(
        df["접수일자"],
        format="mixed",
        errors="raise",
    ).dt.normalize()

    df["접수지역"] = (
        df["접수지역"]
        .astype("string")
        .str.strip()
        .map(nfc)
    )
    df["요일"] = (
        df["요일"]
        .astype("string")
        .str.strip()
        .map(nfc)
    )

    if df["접수지역"].isna().any() or (df["접수지역"] == "").any():
        raise ValueError(f"{nfc(path.name)}: 접수지역 결측 또는 빈 문자열 존재")

    expected_weekday = df["접수일자"].dt.dayofweek.map(KOREAN_WEEKDAY)
    weekday_mismatch = df["요일"] != expected_weekday
    if weekday_mismatch.any():
        examples = df.loc[
            weekday_mismatch, ["접수일자", "요일"]
        ].head().to_dict("records")
        raise ValueError(
            f"{nfc(path.name)}: 접수일자와 요일 불일치 예시: {examples}"
        )

    volume = pd.to_numeric(df["접수통수"], errors="raise")
    if volume.isna().any():
        raise ValueError(f"{nfc(path.name)}: 접수통수 결측 존재")
    if ((volume % 1) != 0).any():
        raise ValueError(f"{nfc(path.name)}: 접수통수에 소수 값 존재")
    if (volume < 0).any():
        raise ValueError(f"{nfc(path.name)}: 접수통수에 음수 존재")
    df["접수통수"] = volume.astype("int64")

    for column in EVENT_COLUMNS:
        values = pd.to_numeric(df[column], errors="raise")
        if values.isna().any():
            raise ValueError(f"{nfc(path.name)}: {column} 결측 존재")
        if not values.isin([0, 1]).all():
            invalid_values = sorted(values.loc[~values.isin([0, 1])].unique())
            raise ValueError(
                f"{nfc(path.name)}: {column}에 0/1 이외 값 존재: {invalid_values}"
            )
        df[column] = values.astype("int8")

    file_yyyymm = extract_yyyymm(path.name)
    if file_yyyymm is None:
        raise ValueError(f"{nfc(path.name)}: 파일명에서 YYYYMM을 찾지 못함")
    month_mismatch = df["접수일자"].dt.strftime("%Y%m") != file_yyyymm
    if month_mismatch.any():
        raise ValueError(
            f"{nfc(path.name)}: 파일 연월과 접수일자 불일치 "
            f"{int(month_mismatch.sum())}건"
        )

    cleaned_frames.append(df)
    processing_rows.append(
        {
            "파일연월": file_yyyymm,
            "파일명": nfc(path.name),
            "인코딩": encoding,
            "원본행수": len(raw_df),
            "제거한완전공백행": int(blank_mask.sum()),
            "제거한Unnamed컬럼": ", ".join(unnamed_columns) or None,
            "정제후행수": len(df),
            "최소날짜": df["접수일자"].min().date().isoformat(),
            "최대날짜": df["접수일자"].max().date().isoformat(),
        }
    )

processing_log = pd.DataFrame(processing_rows)
display(processing_log)

,파일연월,파일명,인코딩,원본행수,제거한완전공백행,제거한Unnamed컬럼,정제후행수,최소날짜,최대날짜
0,202401,대전광역시_일반통상정보_202401_라벨링.csv,utf-8-sig,22,0,NaN,22,2024-01-02,2024-01-31
1,202402,대전광역시_일반통상정보_202402_라벨링.csv,utf-8-sig,25,6,NaN,19,2024-02-01,2024-02-29
2,202403,대전광역시_일반통상정보_202403_라벨링.csv,utf-8-sig,22,0,NaN,22,2024-03-04,2024-03-31
3,202404,대전광역시_일반통상정보_202404_라벨링.csv,utf-8-sig,25,4,Unnamed: 11,21,2024-04-01,2024-04-30
4,202405,대전광역시_일반통상정보_202405_라벨링.csv,utf-8-sig,22,1,NaN,21,2024-05-01,2024-05-31
5,202406,대전광역시_일반통상정보_202406_라벨링.csv,utf-8-sig,19,0,NaN,19,2024-06-03,2024-06-28
6,202407,대전광역시_일반통상정보_202407_라벨링.csv,utf-8-sig,24,1,NaN,23,2024-07-01,2024-07-31
7,202408,대전광역시_일반통상정보_202408_라벨링.csv,utf-8-sig,21,0,NaN,21,2024-08-01,2024-08-30
8,202409,대전광역시_일반통상정보_202409_라벨링.csv,utf-8-sig,18,0,NaN,18,2024-09-02,2024-09-30
9,202410,대전광역시_일반통상정보_202410_라벨링.csv,utf-8-sig,20,0,NaN,20,2024-10-02,2024-10-31


## 3. 병합 후 무결성 검증

월별 정제 결과를 날짜순으로 결합한다. 날짜 중복, 결측치, 스키마, 자료형을 확인하고 누락 날짜는 표시만 하며 임의로 채우지 않는다.

In [4]:
combined = (
    pd.concat(cleaned_frames, ignore_index=True)
    .sort_values("접수일자")
    .reset_index(drop=True)
)

duplicate_dates = combined.loc[
    combined["접수일자"].duplicated(keep=False), "접수일자"
]
if not duplicate_dates.empty:
    raise ValueError(
        "병합 후 중복 날짜가 존재합니다: "
        + ", ".join(duplicate_dates.dt.strftime("%Y-%m-%d").unique()[:20])
    )

if list(combined.columns) != CANONICAL_COLUMNS:
    raise AssertionError("최종 컬럼명 또는 순서가 기준과 다릅니다.")

if combined.isna().any().any():
    missing_by_column = combined.isna().sum()
    missing_by_column = missing_by_column[missing_by_column > 0].to_dict()
    raise ValueError(f"정제 결과에 결측값 존재: {missing_by_column}")

expected_dtypes = {
    "접수일자": "datetime64[us]",
    "접수지역": "string",
    "접수통수": "int64",
    "요일": "string",
}
for column in EVENT_COLUMNS:
    expected_dtypes[column] = "int8"

actual_dtypes = {column: str(dtype) for column, dtype in combined.dtypes.items()}

full_calendar = pd.date_range(
    combined["접수일자"].min(),
    combined["접수일자"].max(),
    freq="D",
)
missing_calendar_dates = full_calendar.difference(combined["접수일자"])

validation_summary = pd.DataFrame(
    [
        {"검증항목": "입력 파일", "결과": len(monthly_files)},
        {"검증항목": "원본 행 합계", "결과": int(processing_log["원본행수"].sum())},
        {
            "검증항목": "제거한 완전 공백 행",
            "결과": int(processing_log["제거한완전공백행"].sum()),
        },
        {"검증항목": "최종 행 수", "결과": len(combined)},
        {"검증항목": "최종 컬럼 수", "결과": len(combined.columns)},
        {"검증항목": "중복 날짜", "결과": int(combined["접수일자"].duplicated().sum())},
        {"검증항목": "결측 셀", "결과": int(combined.isna().sum().sum())},
        {
            "검증항목": "최소 날짜",
            "결과": combined["접수일자"].min().date().isoformat(),
        },
        {
            "검증항목": "최대 날짜",
            "결과": combined["접수일자"].max().date().isoformat(),
        },
        {
            "검증항목": "캘린더상 누락 날짜",
            "결과": len(missing_calendar_dates),
        },
    ]
)

display(validation_summary)
display(pd.DataFrame({"컬럼": combined.columns, "자료형": combined.dtypes.astype(str)}))
display(combined.head())
display(combined.tail())

,검증항목,결과
0,입력 파일,30
1,원본 행 합계,632
2,제거한 완전 공백 행,15
3,최종 행 수,617
4,최종 컬럼 수,11
5,중복 날짜,0
6,결측 셀,0
7,최소 날짜,2024-01-02
8,최대 날짜,2026-06-30
9,캘린더상 누락 날짜,294


,컬럼,자료형
접수일자,접수일자,datetime64[us]
접수지역,접수지역,str
접수통수,접수통수,int64
요일,요일,str
제1기분 자동차세,제1기분 자동차세,int8
재산세(건축),재산세(건축),int8
정기분 주민세,정기분 주민세,int8
주민세(사업소분),주민세(사업소분),int8
재산세(토지),재산세(토지),int8
제2기분 자동차세,제2기분 자동차세,int8


,접수일자,접수지역,접수통수,요일,제1기분 자동차세,재산세(건축),정기분 주민세,주민세(사업소분),재산세(토지),제2기분 자동차세,사회보험료 통합
0,2024-01-02,대전광역시,85267,화,0,0,0,0,0,0,0
1,2024-01-03,대전광역시,118718,수,0,0,0,0,0,0,0
2,2024-01-04,대전광역시,93382,목,0,0,0,0,0,0,0
3,2024-01-05,대전광역시,57788,금,0,0,0,0,0,0,0
4,2024-01-08,대전광역시,81652,월,0,0,0,0,0,0,0


,접수일자,접수지역,접수통수,요일,제1기분 자동차세,재산세(건축),정기분 주민세,주민세(사업소분),재산세(토지),제2기분 자동차세,사회보험료 통합
612,2026-06-24,대전광역시,22315,수,0,0,0,0,0,0,1
613,2026-06-25,대전광역시,78043,목,0,0,0,0,0,0,1
614,2026-06-26,대전광역시,35848,금,0,0,0,0,0,0,0
615,2026-06-29,대전광역시,38909,월,0,0,0,0,0,0,0
616,2026-06-30,대전광역시,48869,화,0,0,0,0,0,0,0


## 4. 이벤트 라벨 분포 확인

이벤트 컬럼별 `1`의 개수를 확인한다. 이 단계에서는 라벨을 추가하거나 수정하지 않는다.

In [5]:
event_summary = pd.DataFrame(
    {
        "이벤트": EVENT_COLUMNS,
        "0_개수": [(combined[column] == 0).sum() for column in EVENT_COLUMNS],
        "1_개수": [(combined[column] == 1).sum() for column in EVENT_COLUMNS],
    }
)
event_summary["1_비율"] = (
    event_summary["1_개수"] / len(combined)
).round(4)
display(event_summary)

,이벤트,0_개수,1_개수,1_비율
0,제1기분 자동차세,602,15,0.0243
1,재산세(건축),602,15,0.0243
2,정기분 주민세,607,10,0.0162
3,주민세(사업소분),607,10,0.0162
4,재산세(토지),612,5,0.0081
5,제2기분 자동차세,607,10,0.0162
6,사회보험료 통합,509,108,0.1750


## 5. Parquet·CSV 저장 및 재검증

모든 검증을 통과한 경우에만 같은 통합 데이터를 Parquet과 UTF-8 BOM CSV로 저장한다.
두 파일을 다시 읽어 행 수, 컬럼, 값, 날짜 중복과 결측치를 재확인한다.
마지막으로 원본 CSV의 SHA-256 해시가 실행 전과 동일한지 검사한다.

In [6]:
CSV_OUTPUT_PATH = OUTPUT_PATH.with_suffix(".csv")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
combined.to_parquet(OUTPUT_PATH, index=False)
combined.to_csv(
    CSV_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d",
)

parquet_reloaded = pd.read_parquet(OUTPUT_PATH)
csv_reloaded = pd.read_csv(
    CSV_OUTPUT_PATH,
    encoding="utf-8-sig",
    parse_dates=["접수일자"],
)

for output_name, reloaded in (
    ("Parquet", parquet_reloaded),
    ("CSV", csv_reloaded),
):
    if len(reloaded) != len(combined):
        raise AssertionError(f"{output_name} 저장 후 행 수가 달라졌습니다.")
    if list(reloaded.columns) != CANONICAL_COLUMNS:
        raise AssertionError(
            f"{output_name} 저장 후 컬럼명 또는 순서가 달라졌습니다."
        )
    if reloaded["접수일자"].duplicated().any():
        raise AssertionError(f"{output_name} 결과에 중복 날짜가 존재합니다.")
    if reloaded.isna().any().any():
        raise AssertionError(f"{output_name} 결과에 결측값이 존재합니다.")

pd.testing.assert_frame_equal(
    csv_reloaded[CANONICAL_COLUMNS],
    combined[CANONICAL_COLUMNS],
    check_dtype=False,
)

raw_hashes_after = {str(path): file_hash(path) for path in monthly_files}
if raw_hashes_before != raw_hashes_after:
    changed_files = [
        path
        for path in raw_hashes_before
        if raw_hashes_before[path] != raw_hashes_after[path]
    ]
    raise AssertionError(f"원본 파일 해시 변경 감지: {changed_files}")

final_summary = pd.DataFrame(
    [
        {
            "항목": "Parquet 출력",
            "결과": str(OUTPUT_PATH.relative_to(PROJECT_ROOT)),
        },
        {
            "항목": "CSV 출력",
            "결과": str(CSV_OUTPUT_PATH.relative_to(PROJECT_ROOT)),
        },
        {"항목": "행 수", "결과": len(csv_reloaded)},
        {"항목": "컬럼 수", "결과": len(csv_reloaded.columns)},
        {
            "항목": "날짜 범위",
            "결과": (
                f"{csv_reloaded['접수일자'].min().date()} ~ "
                f"{csv_reloaded['접수일자'].max().date()}"
            ),
        },
        {
            "항목": "중복 날짜",
            "결과": int(csv_reloaded["접수일자"].duplicated().sum()),
        },
        {
            "항목": "결측 셀",
            "결과": int(csv_reloaded.isna().sum().sum()),
        },
        {"항목": "CSV 인코딩", "결과": "utf-8-sig"},
        {"항목": "원본 변경", "결과": "없음"},
    ]
)

display(final_summary)
print("Parquet 및 CSV 저장 완료")

,항목,결과
0,Parquet 출력,data/interim/daily_volume_event_clean.parquet
1,CSV 출력,data/interim/daily_volume_event_clean.csv
2,행 수,617
3,컬럼 수,11
4,날짜 범위,2024-01-02 ~ 2026-06-30
5,중복 날짜,0
6,결측 셀,0
7,CSV 인코딩,utf-8-sig
8,원본 변경,없음


Parquet 및 CSV 저장 완료
